# Paper Check Log Analysis

Этот ноутбук не запускает новые симуляции. Он читает уже готовые логи stage 2 из нескольких machine-root директорий, склеивает их в один каталог trial-ов, дедуплицирует по `trial_idx`, строит group-level pairing `optimized` vs `random` и считает основные статистические тесты.

По умолчанию ноутбук ожидает две директории с логами:

- `experiments/paper_check_flow_lenia/checkpoints`
- `experiments/paper_check_flow_lenia/checkpoints3`

Если у тебя пути другие, поменяй список `RESULT_ROOTS` в следующей ячейке.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur] + list(cur.parents):
        if (candidate / '.git').exists() or (candidate / 'scripts').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd())
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)
sns.set_theme(context='notebook', style='whitegrid')

REPO_ROOT


In [ ]:
RESULT_ROOTS = [
    REPO_ROOT / 'experiments/paper_check_flow_lenia/checkpoints',
    REPO_ROOT / 'experiments/paper_check_flow_lenia/checkpoints3',
]

OUTPUT_DIR = REPO_ROOT / 'analysis/results/paper_check_log_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Как агрегировать M random baseline-ов внутри одного optimized_run_idx
RANDOM_AGG = 'mean'  # options: 'mean', 'median'

PRIMARY_METRICS = [
    'baseline_distance',
    'walls_effect_distance',
    'effect_minus_baseline',
    'effect_over_baseline_ratio',
    'clip_oe_loss_walls_minus_control_mean',
    'msc_score_walls_minus_control_mean',
    'msc_loss_walls_minus_control_mean',
]

# Нули/бейслайны для one-sample tests.
NULL_HYPOTHESES = {
    'effect_minus_baseline': 0.0,
    'effect_over_baseline_ratio': 1.0,
    'clip_oe_loss_walls_minus_control_mean': 0.0,
    'msc_score_walls_minus_control_mean': 0.0,
    'msc_loss_walls_minus_control_mean': 0.0,
}

# Для этих метрик положительное значение интерпретируем как "сильнее эффект".
GREATER_IS_STRONGER = {
    'baseline_distance',
    'walls_effect_distance',
    'effect_minus_baseline',
    'effect_over_baseline_ratio',
    'clip_oe_loss_walls_minus_control_mean',
    'msc_score_walls_minus_control_mean',
}

RESULT_ROOTS


In [ ]:
def resolve_frustration_root(root: Path) -> Path:
    root = Path(root)
    if (root / 'trial_results.csv').exists() or (root / 'trial_data').exists():
        return root
    if (root / 'frustration_simulation').exists():
        return root / 'frustration_simulation'
    return root


def load_trial_rows(root: Path) -> pd.DataFrame:
    fs_root = resolve_frustration_root(root)
    csv_path = fs_root / 'trial_results.csv'
    summary_path = fs_root / 'summary.json'
    rows = []
    source_summary = None

    if summary_path.exists():
        source_summary = json.loads(summary_path.read_text())

    if csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        trial_data_dir = fs_root / 'trial_data'
        for path in sorted(trial_data_dir.glob('trial_*.json')):
            rows.append(json.loads(path.read_text()))
        df = pd.DataFrame(rows)

    if df.empty:
        return df

    df = df.copy()
    df['source_root'] = str(root)
    df['source_name'] = Path(root).name
    df['frustration_root'] = str(fs_root)
    if source_summary is not None:
        for key, value in source_summary.items():
            if key not in df.columns:
                df[f'summary__{key}'] = value
    return df



def coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object:
            try:
                out[col] = pd.to_numeric(out[col])
            except Exception:
                pass
    return out



def bh_fdr(p_values) -> list[float]:
    vals = [float(p) if p is not None and not pd.isna(p) else np.nan for p in p_values]
    n = sum(not np.isnan(v) for v in vals)
    if n == 0:
        return [np.nan] * len(vals)
    order = np.argsort([np.inf if np.isnan(v) else v for v in vals])
    ranked = np.array([vals[i] for i in order], dtype=float)
    adj = np.full(len(vals), np.nan, dtype=float)
    running = 1.0
    valid_seen = 0
    for rev_idx in range(len(order) - 1, -1, -1):
        idx = order[rev_idx]
        p = vals[idx]
        if np.isnan(p):
            continue
        valid_seen += 1
        rank = n - valid_seen + 1
        candidate = min(1.0, p * n / rank)
        running = min(running, candidate)
        adj[idx] = running
    return adj.tolist()



def bootstrap_mean_ci(x, *, n_boot=20000, seed=0, alpha=0.05):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, x.size, size=(n_boot, x.size))
    boot = x[idx].mean(axis=1)
    return tuple(np.quantile(boot, [alpha / 2, 1 - alpha / 2]))



def two_sided_ttest_1samp(x, null_value):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return np.nan
    try:
        return float(stats.ttest_1samp(x, popmean=null_value, alternative='two-sided').pvalue)
    except Exception:
        return np.nan



def two_sided_ttest_rel(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 2:
        return np.nan
    try:
        return float(stats.ttest_rel(x, y, alternative='two-sided').pvalue)
    except Exception:
        return np.nan



def one_sample_test(series, *, null_value=0.0, alternative='greater'):
    x = pd.Series(series).dropna().astype(float).to_numpy()
    centered = x - float(null_value)
    centered_nz = centered[np.abs(centered) > 1e-12]
    gt = int(np.sum(centered > 0))
    lt = int(np.sum(centered < 0))
    n = int(x.size)
    n_nz = int(centered_nz.size)
    ci_low, ci_high = bootstrap_mean_ci(centered, seed=0)
    result = {
        'n': n,
        'null_value': float(null_value),
        'mean': float(np.mean(x)) if n else np.nan,
        'median': float(np.median(x)) if n else np.nan,
        'std': float(np.std(x, ddof=1)) if n > 1 else 0.0,
        'mean_minus_null': float(np.mean(centered)) if n else np.nan,
        'median_minus_null': float(np.median(centered)) if n else np.nan,
        'ci95_mean_minus_null_low': float(ci_low),
        'ci95_mean_minus_null_high': float(ci_high),
        'n_positive': gt,
        'n_negative': lt,
        'wilcoxon_two_sided_p': np.nan,
        'wilcoxon_greater_p': np.nan,
        'sign_two_sided_p': np.nan,
        'sign_greater_p': np.nan,
        'ttest_two_sided_p': two_sided_ttest_1samp(x, null_value),
    }
    if n_nz > 0:
        try:
            result['wilcoxon_two_sided_p'] = float(stats.wilcoxon(centered, alternative='two-sided').pvalue)
        except Exception:
            pass
        try:
            result['wilcoxon_greater_p'] = float(stats.wilcoxon(centered, alternative='greater').pvalue)
        except Exception:
            pass
        try:
            result['sign_two_sided_p'] = float(stats.binomtest(gt, n_nz, 0.5, alternative='two-sided').pvalue)
        except Exception:
            pass
        try:
            result['sign_greater_p'] = float(stats.binomtest(gt, n_nz, 0.5, alternative='greater').pvalue)
        except Exception:
            pass
    return result



def paired_test(x, y, *, alternative='greater'):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)
    mask = x.notna() & y.notna()
    x = x[mask].to_numpy()
    y = y[mask].to_numpy()
    d = x - y
    d_nz = d[np.abs(d) > 1e-12]
    gt = int(np.sum(d > 0))
    lt = int(np.sum(d < 0))
    n = int(d.size)
    n_nz = int(d_nz.size)
    ci_low, ci_high = bootstrap_mean_ci(d, seed=0)
    result = {
        'n_pairs': n,
        'optimized_mean': float(np.mean(x)) if n else np.nan,
        'random_agg_mean': float(np.mean(y)) if n else np.nan,
        'mean_diff': float(np.mean(d)) if n else np.nan,
        'median_diff': float(np.median(d)) if n else np.nan,
        'std_diff': float(np.std(d, ddof=1)) if n > 1 else 0.0,
        'ci95_mean_diff_low': float(ci_low),
        'ci95_mean_diff_high': float(ci_high),
        'n_diff_gt_0': gt,
        'n_diff_lt_0': lt,
        'wilcoxon_two_sided_p': np.nan,
        'wilcoxon_greater_p': np.nan,
        'sign_two_sided_p': np.nan,
        'sign_greater_p': np.nan,
        'ttest_two_sided_p': two_sided_ttest_rel(x, y),
        'cohen_dz': float(np.mean(d) / np.std(d, ddof=1)) if n > 1 and np.std(d, ddof=1) > 0 else np.nan,
    }
    if n_nz > 0:
        try:
            result['wilcoxon_two_sided_p'] = float(stats.wilcoxon(d, alternative='two-sided').pvalue)
        except Exception:
            pass
        try:
            result['wilcoxon_greater_p'] = float(stats.wilcoxon(d, alternative=alternative).pvalue)
        except Exception:
            pass
        try:
            result['sign_two_sided_p'] = float(stats.binomtest(gt, n_nz, 0.5, alternative='two-sided').pvalue)
        except Exception:
            pass
        try:
            result['sign_greater_p'] = float(stats.binomtest(gt, n_nz, 0.5, alternative=alternative).pvalue)
        except Exception:
            pass
    return result



def build_group_level_table(rows: pd.DataFrame, metric_cols: list[str], random_agg='mean') -> pd.DataFrame:
    optimized = (
        rows.loc[rows['candidate_kind'] == 'optimized']
        .sort_values(['optimized_run_idx', 'trial_idx'])
        .drop_duplicates(subset=['optimized_run_idx'], keep='last')
        .copy()
    )
    randoms = rows.loc[rows['candidate_kind'] == 'random'].copy()
    if optimized.empty:
        raise ValueError('No optimized rows found.')
    if randoms.empty:
        raise ValueError('No random rows found.')

    agg_fn = np.mean if random_agg == 'mean' else np.median
    random_agg_df = randoms.groupby('optimized_run_idx')[metric_cols].agg(agg_fn)
    random_agg_df = random_agg_df.add_suffix(f'__random_{random_agg}')
    random_counts = randoms.groupby('optimized_run_idx').size().rename('n_random')

    paired = optimized.set_index('optimized_run_idx').join(random_agg_df, how='inner').join(random_counts, how='inner')
    paired = paired.reset_index().sort_values('optimized_run_idx').reset_index(drop=True)
    for metric in metric_cols:
        rhs = f'{metric}__random_{random_agg}'
        if rhs in paired.columns:
            paired[f'{metric}__opt_minus_random_{random_agg}'] = paired[metric] - paired[rhs]
    return paired


In [ ]:
loaded_frames = []
missing_roots = []
for root in RESULT_ROOTS:
    root = Path(root)
    if not root.exists():
        missing_roots.append(root)
        continue
    frame = load_trial_rows(root)
    if frame.empty:
        warnings.warn(f'No trial rows found under {root}')
        continue
    loaded_frames.append(frame)

if missing_roots:
    print('Missing roots:')
    for path in missing_roots:
        print('  -', path)

if not loaded_frames:
    raise FileNotFoundError('No paper_check logs found in RESULT_ROOTS. Update the paths in the config cell.')

rows = pd.concat(loaded_frames, ignore_index=True)
rows = coerce_numeric(rows)
rows['trial_idx'] = rows['trial_idx'].astype(int)
rows = rows.sort_values(['trial_idx', 'source_name']).reset_index(drop=True)

duplicate_trial_ids = rows.loc[rows.duplicated(subset=['trial_idx'], keep=False), ['trial_idx', 'source_name', 'candidate_kind', 'candidate_idx']]
if not duplicate_trial_ids.empty:
    display(Markdown('### Duplicate `trial_idx` entries detected before deduplication'))
    display(duplicate_trial_ids.sort_values(['trial_idx', 'source_name']).reset_index(drop=True))

rows = rows.drop_duplicates(subset=['trial_idx'], keep='last').reset_index(drop=True)

numeric_cols = rows.select_dtypes(include=[np.number]).columns.tolist()
metric_cols = [c for c in PRIMARY_METRICS if c in rows.columns]
paired = build_group_level_table(rows, metric_cols=metric_cols, random_agg=RANDOM_AGG)

rows.to_csv(OUTPUT_DIR / 'merged_trial_results.csv', index=False)
paired.to_csv(OUTPUT_DIR / f'group_level_pairs_random_{RANDOM_AGG}.csv', index=False)

print('Merged rows:', rows.shape)
print('Group-level paired rows:', paired.shape)
print('Output dir:', OUTPUT_DIR)


## Sanity Checks

In [ ]:
root_summary = (
    rows.groupby('source_name')
    .agg(
        n_trials=('trial_idx', 'count'),
        n_groups=('optimized_run_idx', 'nunique'),
        n_optimized=('candidate_kind', lambda s: int((s == 'optimized').sum())),
        n_random=('candidate_kind', lambda s: int((s == 'random').sum())),
    )
    .reset_index()
)

group_coverage = (
    rows.groupby(['optimized_run_idx', 'candidate_kind'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .sort_values('optimized_run_idx')
)

display(Markdown('### Per-root coverage'))
display(root_summary)

display(Markdown('### Per-group candidate coverage'))
display(group_coverage)

if 'n_random' in paired.columns:
    display(Markdown('### Random baseline count per optimized group'))
    display(paired[['optimized_run_idx', 'n_random']].sort_values('optimized_run_idx').reset_index(drop=True))


## Descriptive Summaries

In [ ]:
if not metric_cols:
    raise ValueError('None of the primary metric columns were found in merged logs.')

summary_by_kind = (
    rows.groupby('candidate_kind')[metric_cols]
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)
summary_by_kind


In [ ]:
paired_preview_cols = ['optimized_run_idx', 'candidate_label', 'n_random']
for metric in metric_cols:
    rhs = f'{metric}__random_{RANDOM_AGG}'
    diff = f'{metric}__opt_minus_random_{RANDOM_AGG}'
    for col in (metric, rhs, diff):
        if col in paired.columns:
            paired_preview_cols.append(col)

display(Markdown(f'### Group-level paired table (`optimized` vs `random_{RANDOM_AGG}`)'))
paired[paired_preview_cols].sort_values('optimized_run_idx').reset_index(drop=True)


## Statistical Tests

Ниже две группы тестов:

1. **Within-kind one-sample tests**: проверяем, превышает ли групповой эффект свой null baseline.
   - для `effect_minus_baseline`, `clip_oe_loss_walls_minus_control_mean`, `msc_*_minus_control_mean` null = `0`
   - для `effect_over_baseline_ratio` null = `1`
2. **Paired optimized-vs-random tests**: сравниваем `optimized` c агрегированным `random_mean`/`random_median` внутри одного `optimized_run_idx`.

Для каждой метрики считаются:

- Wilcoxon signed-rank (`two-sided` и `greater`)
- sign / binomial test (`two-sided` и `greater`)
- paired / one-sample t-test (`two-sided`)
- bootstrap 95% CI для среднего эффекта
- BH-FDR поправка на множественные проверки


In [ ]:
within_records = []
for metric, null_value in NULL_HYPOTHESES.items():
    if metric not in paired.columns:
        continue

    optimized_series = paired[metric]
    random_series = paired[f'{metric}__random_{RANDOM_AGG}']

    opt_res = one_sample_test(optimized_series, null_value=null_value, alternative='greater')
    opt_res.update({'sample': 'optimized', 'metric': metric})
    within_records.append(opt_res)

    rand_res = one_sample_test(random_series, null_value=null_value, alternative='greater')
    rand_res.update({'sample': f'random_{RANDOM_AGG}', 'metric': metric})
    within_records.append(rand_res)

within_tests = pd.DataFrame(within_records)
if not within_tests.empty:
    within_tests['wilcoxon_greater_fdr_bh'] = bh_fdr(within_tests['wilcoxon_greater_p'])
    within_tests['wilcoxon_two_sided_fdr_bh'] = bh_fdr(within_tests['wilcoxon_two_sided_p'])
    within_tests['sign_greater_fdr_bh'] = bh_fdr(within_tests['sign_greater_p'])
    within_tests['sign_two_sided_fdr_bh'] = bh_fdr(within_tests['sign_two_sided_p'])
    within_tests = within_tests.sort_values(['metric', 'sample']).reset_index(drop=True)

within_tests.to_csv(OUTPUT_DIR / f'within_kind_tests_random_{RANDOM_AGG}.csv', index=False)
within_tests


In [ ]:
paired_records = []
for metric in metric_cols:
    rhs = f'{metric}__random_{RANDOM_AGG}'
    if rhs not in paired.columns:
        continue
    res = paired_test(
        paired[metric],
        paired[rhs],
        alternative='greater' if metric in GREATER_IS_STRONGER else 'two-sided',
    )
    res.update({'metric': metric, 'random_aggregation': RANDOM_AGG})
    paired_records.append(res)

paired_tests = pd.DataFrame(paired_records)
if not paired_tests.empty:
    paired_tests['wilcoxon_greater_fdr_bh'] = bh_fdr(paired_tests['wilcoxon_greater_p'])
    paired_tests['wilcoxon_two_sided_fdr_bh'] = bh_fdr(paired_tests['wilcoxon_two_sided_p'])
    paired_tests['sign_greater_fdr_bh'] = bh_fdr(paired_tests['sign_greater_p'])
    paired_tests['sign_two_sided_fdr_bh'] = bh_fdr(paired_tests['sign_two_sided_p'])
    paired_tests = paired_tests.sort_values('metric').reset_index(drop=True)

paired_tests.to_csv(OUTPUT_DIR / f'optimized_vs_random_tests_random_{RANDOM_AGG}.csv', index=False)
paired_tests


## Figures

In [ ]:
plot_metrics = [
    metric for metric in [
        'effect_minus_baseline',
        'effect_over_baseline_ratio',
        'clip_oe_loss_walls_minus_control_mean',
        'msc_score_walls_minus_control_mean',
        'msc_loss_walls_minus_control_mean',
    ]
    if metric in paired.columns and f'{metric}__random_{RANDOM_AGG}' in paired.columns
]

if plot_metrics:
    ncols = 2
    nrows = math.ceil(len(plot_metrics) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.8 * nrows), squeeze=False)
    axes = axes.ravel()

    for ax, metric in zip(axes, plot_metrics):
        rhs = f'{metric}__random_{RANDOM_AGG}'
        plot_df = paired[['optimized_run_idx', metric, rhs]].dropna().copy()
        for _, row in plot_df.iterrows():
            ax.plot(['random', 'optimized'], [row[rhs], row[metric]], color='0.75', linewidth=1)
        ax.scatter(np.repeat('random', len(plot_df)), plot_df[rhs], s=45, label=f'random_{RANDOM_AGG}')
        ax.scatter(np.repeat('optimized', len(plot_df)), plot_df[metric], s=45, label='optimized')
        ax.set_title(metric)
        ax.set_xlabel('')
        ax.legend(loc='best')

    for ax in axes[len(plot_metrics):]:
        ax.axis('off')

    fig.suptitle(f'Per-group paired comparison: optimized vs random_{RANDOM_AGG}', y=1.02, fontsize=14)
    fig.tight_layout()
    plt.show()
else:
    display(Markdown('Нет доступных метрик для paired plots.'))


## Saved Artifacts

После запуска ноутбук сохраняет промежуточные таблицы в `analysis/results/paper_check_log_analysis`:

- `merged_trial_results.csv`
- `group_level_pairs_random_mean.csv` или `group_level_pairs_random_median.csv`
- `within_kind_tests_random_mean.csv` или `within_kind_tests_random_median.csv`
- `optimized_vs_random_tests_random_mean.csv` или `optimized_vs_random_tests_random_median.csv`
